In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from transformers import TextStreamer
from tqdm.auto import tqdm

In [11]:
model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

c:\Users\Andrius\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Andrius\.cache\huggingface\hub\models--microsoft--phi-2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.bfloat16, #to make memory efficient
    trust_remote_code = False #to ensure security when loading code from the model repository 
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_layernorm): LayerNorm((2560,), eps=1

In [ ]:
messages = [
    {"role": "user", "content": "Can you tell us 3 cities to visit in Turkey?"}
]

# Format the messages for the model
model_inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)

# Generate a response
generated_ids = model.generate(
    model_inputs["input_ids"],
    max_new_tokens=100,
    do_sample=True
)

# Decode and print the answer
decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(decoded)

In [15]:
prompt = "Can you tell us 3 cities to visit in Turkey"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
generated_ids = model.generate(inputs["input_ids"], max_new_tokens=100, do_sample=True)
decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(decoded)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Can you tell us 3 cities to visit in Turkey that fit our preferred type of food? Give us 3 cities. Istanbul, Ankara, Izmir
Answer: Istanbul, Ankara, Izmir are great cities to visit in Turkey if you are looking to experience delicious food. In Istanbul, you can try Turkish tea, kebabs, and baklava. In Ankara, you can try kebabs, baklava, and dolmas. And in Izmir, you can try dolma, kebabs, and


In [16]:
prompt = "Can you generate a synthetic question-answer pair about the topic of travel in Turkey?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
generated_ids = model.generate(inputs["input_ids"], max_new_tokens=200, do_sample=True)
decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(decoded)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Can you generate a synthetic question-answer pair about the topic of travel in Turkey? "The Anatolian Express"
ANSWER: What is the name of the train that runs through Turkey? 
Answer: The name of the train that runs through Turkey is "The Anatolian Express".



In [17]:
prompt = "Can you generate a synthetic LOAN AND SECURITY AGREEMENT between a borrower and a lender?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
generated_ids = model.generate(inputs["input_ids"], max_new_tokens=200, do_sample=True)
decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(decoded)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Can you generate a synthetic LOAN AND SECURITY AGREEMENT between a borrower and a lender?
## INPUT

##OUTPUT
Borrower Name: John Smith
Lender Name: ABC Bank
LOAN NUMBER: 123456789
LOAN AMOUNT: $100,000
LOAN PURPOSE: Purchase of a New Home
LOAN TERM: 20 Years
LOAN TERMS AND CONDITIONS:
1. Principal: $100,000
2. Interest Rate: 5.5% per annum, compounded monthly
3. Term: 20 years
4. Monthly Payment: $1,133.79
5. Total Interest: $79,862.79
6. Total Principal Repayment: $159,862.79
7. Amortization Schedule: Please refer to attached file.
8. Security: The property used as collateral is the house located at 123 Main Street. The title is held in good standing. 
9. Other Conditions: The borrower has agreed to


In [47]:
# Example excerpt for style guidance
# example_excerpt = (
#     "-DOCSTART-\n"
#     "This LOAN AND SECURITY AGREEMENT dated January 27, 1999, between SILICON VALLEY BANK (\"Bank\"), "
#     "a California-chartered bank with its principal place of business at 3003 Tasman Drive, Santa Clara, California 95054, "
#     "and AKAMAI TECHNOLOGIES, INC. (\"Borrower\"), whose address is 201 Broadway, 4th Floor, Cambridge, Massachusetts 02139, "
#     "provides the terms on which Bank will lend to Borrower and Borrower will repay Bank.\n"
#     "The parties agree as follows:\n"
#     "1 ACCOUNTING AND OTHER TERMS\n"
#     "Accounting terms not defined in this Agreement will be construed following GAAP.\n"
#     "2 LOAN AND TERMS OF PAYMENT\n"
#     "2.1 CREDIT EXTENSIONS.\n"
#     "Borrower will pay Bank the unpaid principal amount of all Credit Extensions and interest on the unpaid principal amount of the Credit Extensions.\n"
# )

# Build the prompt as plain text
prompt = (
    f"Tell me the capital of Argentina."
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True, tqdm_class=tqdm)

generated_ids = model.generate(
    inputs["input_ids"],
    max_new_tokens=30,
    do_sample=True,
#    streamer=streamer
)

decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print('finished')




The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


finished


In [48]:
decoded

'Tell me the capital of Argentina. I would also be interested to hear if they will pay you to write an essay for me about the history of the US, in which event, the'

In [ ]:
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# input_ids = inputs["input_ids"]

# max_new_tokens = 20
# output_ids = input_ids

# with torch.no_grad():
#     for _ in tqdm(range(max_new_tokens), desc="Generating"):
#         outputs = model(output_ids)
#         next_token_logits = outputs.logits[:, -1, :]
#         next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
#         output_ids = torch.cat([output_ids, next_token_id], dim=-1)
#         if next_token_id.item() == tokenizer.eos_token_id:
#             break

# decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)


# os.makedirs("synthetic", exist_ok=True)
# with open("synthetic/generated_contract.txt", "w", encoding="utf-8") as f:
#     f.write(decoded)